# Pipeline Inspection: Parameter Extraction Pipeline (TomoSAR Elevation Profile Fitting)

This notebook executes the `ParamExtractionPipeline` (`pipelines/param_pipeline/pipeline.py`) stage by stage and verifies that each step produces the contracted output before the next stage consumes it.

The pipeline takes a tomographic SAR cube of shape $(H, A_z, R)$ — backscatter intensity sampled at $H$ discrete elevation (height) bins for every azimuth $A_z$ / range $R$ pixel — and fits, per pixel, a mixture of up to $k_{\max}$ Gaussians to the one-dimensional elevation profile $y(h)$. Each Gaussian is one **scatterer** along height: its centroid $\mu_k$ is the scatterer height, its amplitude $A_k$ the scatterer backscatter strength, and its spread $\sigma_k$ the scatterer extent along elevation. The number of scatterers $K$ active at a pixel is chosen per pixel by a penalised model-selection rule.

The orchestrated sequence, with the nested `ParameterExtractor.run()` → `SigmaFittingExtractor.run()` expanded into its real sub-steps, is:

1. **Peak Initialisation** — `PeakInitialiser.run` seeds $k_{\max}$ peak heights and amplitudes per profile from a smoothed profile.
2. **Sigma Kernel Construction** — `SigmaAdamKernel` / `PmapSigmaAdamKernel` compile the JAX Adam sigma-fitting kernel.
3. **Multi-$K$ Sigma Fitting** — the kernel optimises the spreads $\sigma$ for every candidate $K \in \{1,\dots,k_{\max}\}$ with amplitudes and centroids frozen.
4. **Best-$K$ Selection** — `BestKSelector.select` scores each $K$ by penalised MSE and assembles the winning parameter vector per pixel; `ParameterExtractor._sort_gaussians` orders components by height.
5. **Fitting Metrics** — `FittingMetricsCalculator.run` computes the per-pixel $R^2$ map, the active-Gaussian count map, and per-Gaussian parameter maps.
6. **Artifact I/O** — `ParameterIO.save_params` and `ExtractionMetadataManager.save_run_metadata` persist the parameter stack and the run metadata.

Every stage below is preceded by a markdown header (callable, input, output, governing equations, and a quantitative "what you should see"), then an execution cell, a structure-inspection cell, a publication-quality visualization, a PASS/FAIL assertion cell, and a domain-specific common-mistakes table.

> This notebook is a *verification harness*: run it once to establish a baseline, change a component, run it again, and the assertions either still pass or pinpoint exactly where the contract broke.

In [ ]:
from __future__ import annotations

import sys
import json
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from configuration.param_extraction_config import ExtractionConfig, FitSettings, FitMode
from pipelines.param_pipeline.pipeline      import ParamExtractionPipeline
from pipelines.param_pipeline.peak_init     import PeakInitialiser
from pipelines.param_pipeline.sigma_kernels import SigmaAdamKernel, PmapSigmaAdamKernel, SigmaScan
from pipelines.param_pipeline.sigma_fitting import SigmaFittingExtractor
from pipelines.param_pipeline.fitting       import ParameterExtractor
from pipelines.param_pipeline.best_k        import BestKSelector
from pipelines.param_pipeline.metrics       import FittingMetricsCalculator
from pipelines.param_pipeline.artifact_io   import ParameterIO
from pipelines.param_pipeline.metadata      import ExtractionMetadataManager
from pipelines.param_pipeline.plots         import FittingResultPlotter
from tools.gaussian_mixture                 import GaussianMixture
from tools.logger                           import Logger

import jax
import jax.numpy as jnp

mpl.rcParams.update({
    "font.family"        : "serif",
    "font.size"          : 11,
    "axes.labelsize"     : 12,
    "axes.titlesize"     : 13,
    "legend.fontsize"    : 10,
    "xtick.labelsize"    : 10,
    "ytick.labelsize"    : 10,
    "figure.dpi"         : 150,
    "savefig.dpi"        : 300,
    "savefig.bbox"       : "tight",
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
})

figure_directory = Path("figures/inspect_param_pipeline")
figure_directory.mkdir(parents=True, exist_ok=True)

component_palette = {
    "data"       : "#000000",
    "fit"        : "#d62728",
    "residual"   : "#7f7f7f",
    "g1"         : "#1f77b4",
    "g2"         : "#ff7f0e",
    "g3"         : "#2ca02c",
    "g4"         : "#9467bd",
    "g5"         : "#8c564b",
}

extraction_configuration = ExtractionConfig(
    processed_data_path = project_root / "data" / "processed",
    output_prefix       = "params",
    fit_settings        = FitSettings(fit_config=FitMode.SigmaOnly()),
)

k_max                 = extraction_configuration.fit_settings.fit_config.k_max
lambda_k              = extraction_configuration.fit_settings.fit_config.lambda_k
prominence_fraction   = extraction_configuration.fit_settings.fit_config.prominence_frac
threshold_factor      = extraction_configuration.fit_settings.fit_config.threshold_factor
truncation_index      = extraction_configuration.fit_settings.fit_config.truncation_index
parameters_per_pixel  = extraction_configuration.fit_settings.parameters_per_profile

print("Project root           :", project_root)
print("Processed data path     :", extraction_configuration.processed_data_path)
print("Output directory        :", extraction_configuration.output_directory)
print("k_max (max scatterers)  :", k_max)
print("lambda_k (complexity)   :", lambda_k)
print("prominence fraction     :", prominence_fraction)
print("threshold factor        :", threshold_factor)
print("truncation index        :", truncation_index)
print("parameters per pixel    :", parameters_per_pixel)
print("JAX devices             :", jax.devices())
print("Figure directory        :", figure_directory.resolve())

## Pipeline Overview

`ParamExtractionPipeline.run()` orchestrates the following ordered sequence. Each arrow is a handoff of a concrete object from one component to the next. Stages 1–4 live inside `SigmaFittingExtractor._fit_batch`, which is the per-range-batch core of the extraction; Stages 5–6 run once on the assembled parameter stack.

```
Tomogram cube  (H, Az, R)  intensity per height bin
  └─► Stage 1  PeakInitialiser.run()        →  (amps, mus, sigmas)  seed  (N, k_max)
        └─► Stage 2  Sigma*AdamKernel()       →  compiled JAX Adam sigma optimiser
              └─► Stage 3  _fit_all_K()         →  {K: (amps_norm, mus, sigmas_fit)}  K=1..k_max
                    └─► Stage 4  BestKSelector.select() →  best_params  (N, 3*k_max), height-sorted
                          └─► Stage 5  FittingMetricsCalculator.run() →  R2 map, activity map, param maps
                                └─► Stage 6  ParameterIO + Metadata     →  parameters_*.npy + meta.json
```

The single Gaussian-mixture forward model reconstructed at every stage is

$$\hat{y}(h) \;=\; \sum_{k=1}^{K} A_k \, \exp\!\left( -\frac{(h - \mu_k)^2}{2\,\sigma_k^2} \right),$$

with $h$ the elevation (height) coordinate in metres, $A_k$ the amplitude, $\mu_k$ the centroid height, and $\sigma_k$ the spread of the $k$-th scatterer. Only $\sigma_k$ is optimised on the GPU (amplitudes and centroids are fixed by the peak initialiser), which is why the method is named `sigma_only_adam`.

| Stage | Callable | Input | Output |
|-------|----------|-------|--------|
| 1 | `PeakInitialiser.run` | raw profiles `(N, H)`, height axis `(H,)`, `K=k_max` | `(amps, mus, sigmas)` each `(N, k_max)` |
| 2 | `SigmaAdamKernel()` / `PmapSigmaAdamKernel(devices)` | active JAX devices | callable Adam sigma optimiser |
| 3 | `SigmaFittingExtractor._fit_all_K` (kernel calls) | per-`K` inits, normalised profiles | `{K: (amps_norm, mus, sigmas_fit)}` |
| 4 | `BestKSelector.select` | `{K: ...}`, normalised profiles, scales | `best_params (N, 3*k_max)` |
| 5 | `FittingMetricsCalculator.run` | parameter stack `(3*k_max, Az, R)`, metadata, tomogram | metrics dict (R², activity, maps) |
| 6 | `ParameterIO.save_params` + `ExtractionMetadataManager.save_run_metadata` | parameter stack, paths | `.npy` path + `meta.json` path |

For a fully self-contained, fast-running inspection we synthesise a small set of representative elevation profiles below with a known number of scatterers, then drive each real pipeline component on them. Where a real tomogram is available on disk, swap `synthetic_profiles` / `synthetic_height_axis` for a memory-mapped slice of the cube — every assertion still holds.

In [ ]:
class SyntheticTomogramFactory:
    def __init__(self, n_height_bins, height_low, height_high, random_seed):
        self.n_height_bins   = n_height_bins
        self.height_low      = height_low
        self.height_high     = height_high
        self.random_generator = np.random.default_rng(random_seed)
        self.height_axis      = np.linspace(height_low, height_high, n_height_bins, dtype=np.float32)

    def _single_profile(self, n_scatterers):
        profile = np.zeros(self.n_height_bins, dtype=np.float32)
        span    = self.height_high - self.height_low
        for _ in range(n_scatterers):
            amplitude = self.random_generator.uniform(0.4, 1.0)
            centroid  = self.random_generator.uniform(self.height_low + 0.15 * span, self.height_high - 0.15 * span)
            spread    = self.random_generator.uniform(0.04 * span, 0.10 * span)
            profile  += amplitude * np.exp(-((self.height_axis - centroid) ** 2) / (2.0 * spread ** 2))
        noise    = self.random_generator.normal(0.0, 0.015, size=self.n_height_bins).astype(np.float32)
        return np.clip(profile + noise, 0.0, None)

    def build(self, scatterer_counts):
        profiles = np.stack([self._single_profile(count) for count in scatterer_counts], axis=0)
        return profiles.astype(np.float32)


synthetic_height_bins = 200
synthetic_height_low  = 0.0
synthetic_height_high = 60.0

scatterer_count_plan = [1, 1, 2, 2, 2, 3, 3, 4, 5, 1, 2, 3]

synthetic_factory      = SyntheticTomogramFactory(synthetic_height_bins, synthetic_height_low, synthetic_height_high, random_seed=0)
synthetic_height_axis  = synthetic_factory.height_axis
synthetic_profiles     = synthetic_factory.build(scatterer_count_plan)

inspection_logger = Logger(log_dir=str(figure_directory / "logs"), name="param_inspect", level="INFO")

print("Synthetic profile bank      :", synthetic_profiles.shape, synthetic_profiles.dtype)
print("Height axis                 :", synthetic_height_axis.shape, "from", synthetic_height_axis[0], "to", synthetic_height_axis[-1], "m")
print("True scatterer counts       :", scatterer_count_plan)
print("Profile intensity range     :", float(synthetic_profiles.min()), "to", float(synthetic_profiles.max()))

---
## Stage 1: Peak Initialisation

**Callable:** `PeakInitialiser.run(prof_raw, height_axis, K, prominence_frac, n_workers)`
**Input:** raw (non-negative) elevation profiles `prof_raw` of shape $(N, H)$, the height axis `(H,)`, the maximum scatterer count `K = k_max`, and the prominence fraction.
**Output:** three arrays `(amps, mus, sigmas)`, each of shape $(N, k_{\max})$ and dtype `float32`. `amps[n, g]` is the smoothed-profile height at the $g$-th detected peak, `mus[n, g]` is the peak's height coordinate in metres, and `sigmas[n, g]` is a constant initial guess.

The initialiser smooths each profile with a width-5 uniform filter, then runs `scipy.signal.find_peaks` with prominence threshold $p_{\max}\cdot f_{\text{prom}}$ and a minimum inter-peak distance of $\lceil \sigma_{\text{guess}}/\Delta h \rceil$ bins. The shared sigma guess is

$$\sigma_{\text{guess}} \;=\; \max\!\left(2\,\Delta h,\; \frac{h_{\text{span}}}{8\,K}\right), \qquad \Delta h = h_1 - h_0, \quad h_{\text{span}} = h_{H-1} - h_0.$$

When fewer than $K$ prominent peaks exist, remaining slots are filled greedily from the residual; when none exist, the slots are spread uniformly across the height axis.

> **What you should see:** all three arrays have shape `(N, k_max)` = `(len(profiles), 5)`. `mus` lie strictly within the height range `[0, 60] m`. `sigmas` are all equal to the single $\sigma_{\text{guess}}$ value (here $\approx \max(2\Delta h, 60/40) = 1.5$ m). `amps` are non-negative and bounded by each profile's smoothed peak. Profiles built with $C$ true scatterers should place their first $C$ peaks near the true centroids; the surplus slots ($K-C$) fall on noise or uniform fallback positions.

In [ ]:
peak_initialiser           = PeakInitialiser()
initial_amplitudes, initial_centroids, initial_sigmas = peak_initialiser.run(
    prof_raw        = synthetic_profiles,
    height_axis     = synthetic_height_axis,
    K               = k_max,
    prominence_frac = prominence_fraction,
    n_workers       = 1,
)

height_span        = float(synthetic_height_axis[-1] - synthetic_height_axis[0])
height_step        = float(synthetic_height_axis[1] - synthetic_height_axis[0])
expected_sigma_guess = max(2.0 * height_step, height_span / (8.0 * k_max))

In [ ]:
print("initial_amplitudes shape :", initial_amplitudes.shape, initial_amplitudes.dtype)
print("initial_centroids  shape :", initial_centroids.shape,  initial_centroids.dtype)
print("initial_sigmas     shape :", initial_sigmas.shape,     initial_sigmas.dtype)
print()
print("amplitude   min / mean / max :", float(initial_amplitudes.min()), float(initial_amplitudes.mean()), float(initial_amplitudes.max()))
print("centroid    min / mean / max :", float(initial_centroids.min()),  float(initial_centroids.mean()),  float(initial_centroids.max()), "m")
print("sigma       unique values    :", np.unique(initial_sigmas))
print("expected sigma_guess         :", expected_sigma_guess, "m")
print("height step (dh)             :", height_step, "m")
print()
print("Per-profile centroid seeds (m), first profile vs true count:")
for profile_index in range(min(4, synthetic_profiles.shape[0])):
    print(f"  profile {profile_index}  true_K={scatterer_count_plan[profile_index]}  mus={np.round(initial_centroids[profile_index], 2)}")

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axis_left   = axes[0]
profile_idx = 5
smoothed    = np.convolve(synthetic_profiles[profile_idx], np.ones(5) / 5.0, mode="same")
axis_left.plot(synthetic_height_axis, synthetic_profiles[profile_idx], color=component_palette["data"], lw=1.4, label="raw profile")
axis_left.plot(synthetic_height_axis, smoothed, color="#1f77b4", lw=1.1, ls="--", label="width-5 smoothed")
for slot_index in range(k_max):
    centroid_value  = initial_centroids[profile_idx, slot_index]
    amplitude_value = initial_amplitudes[profile_idx, slot_index]
    axis_left.scatter([centroid_value], [amplitude_value], s=60, zorder=5, color=component_palette[f"g{slot_index + 1}"], edgecolors="black", linewidths=0.6, label=f"seed $g_{{{slot_index + 1}}}$")
axis_left.set_xlabel(r"height $h$ [m]")
axis_left.set_ylabel(r"backscatter intensity")
axis_left.set_title(f"Stage 1 — Peak Initialisation: seeded peaks (profile {profile_idx}, true $K$={scatterer_count_plan[profile_idx]})")
axis_left.legend(fontsize=8, ncol=2)
axis_left.grid(True, lw=0.3, alpha=0.4)

axis_right = axes[1]
centroid_values_flat = initial_centroids.reshape(-1)
axis_right.hist(centroid_values_flat, bins=24, range=(synthetic_height_low, synthetic_height_high), color="#2ca02c", alpha=0.7, edgecolor="white", linewidth=0.5)
axis_right.axvline(centroid_values_flat.mean(), color="black", lw=1.2, ls="--", label=f"mean = {centroid_values_flat.mean():.1f} m")
axis_right.set_xlabel(r"seeded centroid height $\mu_k$ [m]")
axis_right.set_ylabel("count")
axis_right.set_title("Stage 1 — Peak Initialisation: distribution of seeded centroids")
axis_right.legend(fontsize=9)
axis_right.grid(True, axis="y", lw=0.3, alpha=0.4)

figure.tight_layout()
figure.savefig(figure_directory / "stage_1_peak_init.pdf")
plt.show()

In [ ]:
stage_1_checks = [
    ("amps shape is (N, k_max)",           initial_amplitudes.shape == (synthetic_profiles.shape[0], k_max)),
    ("mus shape is (N, k_max)",            initial_centroids.shape  == (synthetic_profiles.shape[0], k_max)),
    ("sigmas shape is (N, k_max)",         initial_sigmas.shape     == (synthetic_profiles.shape[0], k_max)),
    ("all arrays float32",                 initial_amplitudes.dtype == np.float32 and initial_centroids.dtype == np.float32 and initial_sigmas.dtype == np.float32),
    ("no NaN or Inf in seeds",             np.isfinite(initial_amplitudes).all() and np.isfinite(initial_centroids).all() and np.isfinite(initial_sigmas).all()),
    ("amplitudes non-negative",            float(initial_amplitudes.min()) >= 0.0),
    ("centroids within height range",      float(initial_centroids.min()) >= synthetic_height_low and float(initial_centroids.max()) <= synthetic_height_high),
    ("sigmas equal shared guess",          bool(np.allclose(initial_sigmas, expected_sigma_guess))),
]
for description, condition in stage_1_checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 1: Peak Initialisation

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Every profile seeds exactly $K$ evenly spaced centroids regardless of data | Profile maximum below $10^{-10}$, so the uniform-fallback branch fires | Print `synthetic_profiles.max(axis=1)`; check for near-zero (inactive) profiles reaching this stage |
| Seeded centroids collapse onto one or two heights | `prominence_frac` too high, suppressing real peaks; or `min_dist` too large for closely spaced scatterers | Compare `np.unique(initial_centroids[n])` against the smoothed profile peaks; lower `prominence_frac` |
| Surplus slots ($K - C$) carry large amplitudes | Greedy residual fill latched onto noise spikes | Inspect `initial_amplitudes` for the surplus slots; they should be small relative to true peaks |
| `mus` outside `[h0, h_(H-1)]` | Height axis passed in wrong units or reversed | Print `height_axis[0]`, `height_axis[-1]`; confirm metres and ascending order |
| Seeds ignore a clearly visible scatterer | Width-5 smoothing washed out a narrow peak whose $\sigma <$ one bin | Overlay the smoothed curve; if the peak vanished, the profile is undersampled along height |
| `ProcessPoolExecutor` hangs or pickling error | Worker count mismatched with platform fork semantics inside a notebook | Run with `n_workers=1` (as here) for inspection; parallelism is a throughput-only concern |

**Passing to Stage 2:** `(initial_amplitudes, initial_centroids, initial_sigmas)` — three `float32` arrays of shape `(N, k_max)` holding amplitude, centroid-height and spread seeds; the sigma seeds are the starting point the Adam kernel will optimise.

---
## Stage 2: Sigma Kernel Construction

**Callable:** `SigmaAdamKernel()` (single device) or `PmapSigmaAdamKernel(devices)` (multi-GPU), selected inside `SigmaFittingExtractor.__init__`.
**Input:** the list of active JAX devices (here whatever `jax.devices()` returns).
**Output:** a callable kernel `kernel(sigmas_init, height_axis, profiles, amps, mus, sigma_lower, sigma_upper, n_steps, lr, b1, b2)` that returns optimised sigmas of shape $(N, K)$. It JIT-compiles (or `pmap`-shards) the Adam scan over the sigma-only loss.

The per-pixel loss minimised by the kernel is the mean-squared reconstruction error of the Gaussian mixture against the (normalised) profile, with **only $\sigma$ trainable**:

$$\mathcal{L}(\sigma) \;=\; \frac{1}{H}\sum_{n=0}^{H-1}\left( \sum_{k=1}^{K} A_k \exp\!\Big(-\tfrac{(h_n - \mu_k)^2}{2\sigma_k^2}\Big) - y(h_n) \right)^2,$$

optimised by Adam with bias-corrected moments, $\sigma$ clipped to $[\sigma_{\text{lower}}, \sigma_{\text{upper}}] = [\Delta h,\; h_{\text{span}}/2]$ at every step. We exercise the kernel here with a tiny warm-up batch to force compilation and confirm it runs end to end.

> **What you should see:** a kernel object whose class is `SigmaAdamKernel` when one device is active or `PmapSigmaAdamKernel` when several are. A 2-step warm-up call on a `(4, K)` dummy batch returns sigmas of shape `(4, k_max)`, all finite and clipped within `[sigma_lower, sigma_upper]`. The loss evaluated at the returned sigmas must be a finite non-negative scalar.

In [ ]:
available_gpu_devices = [device for device in jax.devices() if device.platform in ("gpu", "cuda")]
active_devices        = available_gpu_devices if available_gpu_devices else jax.devices()

if len(active_devices) > 1:
    sigma_kernel    = PmapSigmaAdamKernel(active_devices)
    kernel_backend  = f"pmap ({len(active_devices)} devices)"
else:
    sigma_kernel    = SigmaAdamKernel()
    kernel_backend  = "jit (1 device)"

height_axis_jax = jnp.asarray(synthetic_height_axis)
sigma_lower_jax = jnp.float32(height_step)
sigma_upper_jax = jnp.float32(height_span / 2.0)

warmup_batch    = max(len(active_devices), 4)
warmup_sigmas   = jnp.ones((warmup_batch, k_max),  dtype=jnp.float32) * expected_sigma_guess
warmup_profiles = jnp.asarray(synthetic_profiles[:warmup_batch] / np.maximum(synthetic_profiles[:warmup_batch].max(axis=1, keepdims=True), 1e-7))
warmup_amps     = jnp.asarray(initial_amplitudes[:warmup_batch] / np.maximum(synthetic_profiles[:warmup_batch].max(axis=1, keepdims=True), 1e-7))
warmup_mus      = jnp.asarray(initial_centroids[:warmup_batch])

warmup_sigmas_fit = sigma_kernel(
    warmup_sigmas, height_axis_jax, warmup_profiles, warmup_amps, warmup_mus,
    sigma_lower_jax, sigma_upper_jax,
    2, float(extraction_configuration.adam_lr), float(extraction_configuration.adam_b1), float(extraction_configuration.adam_b2),
)
warmup_sigmas_fit = np.asarray(warmup_sigmas_fit, dtype=np.float32)

In [ ]:
warmup_loss = float(SigmaScan.per_pixel_loss(
    jnp.asarray(warmup_sigmas_fit[0]),
    height_axis_jax,
    warmup_profiles[0],
    warmup_amps[0],
    warmup_mus[0],
))

print("kernel class            :", type(sigma_kernel).__name__)
print("kernel backend          :", kernel_backend)
print("active devices          :", active_devices)
print("sigma_lower (dh)        :", float(sigma_lower_jax), "m")
print("sigma_upper (span/2)    :", float(sigma_upper_jax), "m")
print()
print("warmup output shape     :", warmup_sigmas_fit.shape, warmup_sigmas_fit.dtype)
print("warmup sigma min / max  :", float(warmup_sigmas_fit.min()), float(warmup_sigmas_fit.max()), "m")
print("warmup sigmas finite    :", bool(np.isfinite(warmup_sigmas_fit).all()))
print("warmup loss at sigma*   :", warmup_loss)

In [ ]:
figure, axis = plt.subplots(figsize=(8.5, 5.0))

sigma_sweep = np.linspace(float(sigma_lower_jax), float(sigma_upper_jax), 160, dtype=np.float32)
profile_norm_single = warmup_profiles[0]
amps_single = warmup_amps[0]
mus_single  = warmup_mus[0]

loss_curve = np.array([
    float(SigmaScan.per_pixel_loss(jnp.full((k_max,), trial_sigma, dtype=jnp.float32), height_axis_jax, profile_norm_single, amps_single, mus_single))
    for trial_sigma in sigma_sweep
])

axis.plot(sigma_sweep, loss_curve, color="#1f77b4", lw=1.6, label=r"$\mathcal{L}(\sigma)$ (shared $\sigma$ sweep)")
fitted_shared_sigma = float(np.mean(warmup_sigmas_fit[0]))
axis.axvline(expected_sigma_guess, color="#7f7f7f", lw=1.1, ls=":", label=f"init guess = {expected_sigma_guess:.2f} m")
axis.axvline(fitted_shared_sigma, color=component_palette["fit"], lw=1.3, ls="--", label=f"mean fitted $\\sigma$ = {fitted_shared_sigma:.2f} m")
axis.set_xlabel(r"trial spread $\sigma$ [m]")
axis.set_ylabel(r"per-pixel MSE loss $\mathcal{L}(\sigma)$")
axis.set_title("Stage 2 — Sigma Kernel: loss landscape over the shared spread for one profile")
axis.legend(fontsize=9)
axis.grid(True, lw=0.3, alpha=0.4)

figure.tight_layout()
figure.savefig(figure_directory / "stage_2_kernel.pdf")
plt.show()

In [ ]:
stage_2_checks = [
    ("kernel is a known kernel type",       isinstance(sigma_kernel, (SigmaAdamKernel, PmapSigmaAdamKernel))),
    ("kernel is callable",                  callable(sigma_kernel)),
    ("warmup output shape (warmup, k_max)", warmup_sigmas_fit.shape == (warmup_batch, k_max)),
    ("warmup sigmas finite",                bool(np.isfinite(warmup_sigmas_fit).all())),
    ("warmup sigmas >= sigma_lower",        float(warmup_sigmas_fit.min()) >= float(sigma_lower_jax) - 1e-4),
    ("warmup sigmas <= sigma_upper",        float(warmup_sigmas_fit.max()) <= float(sigma_upper_jax) + 1e-4),
    ("warmup loss finite non-negative",     np.isfinite(warmup_loss) and warmup_loss >= 0.0),
    ("sigma_lower equals dh",               abs(float(sigma_lower_jax) - height_step) < 1e-4),
]
for description, condition in stage_2_checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 2: Sigma Kernel Construction

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Fitted sigmas pin to `sigma_upper` ($h_{\text{span}}/2$) | Centroids/amplitudes seeded so far off that a single broad Gaussian minimises MSE | Inspect `mus`/`amps` seeds for that pixel; verify the loss landscape is not monotone toward the upper bound |
| Fitted sigmas pin to `sigma_lower` ($\Delta h$) | Profile is a delta-like spike narrower than one height bin (under-sampled scatterer) | Plot the profile; if it is one bin wide the spread is unresolvable |
| Kernel recompiles on every batch | Batch shape `(N, K)` changes call to call, defeating the JIT cache | Keep `N = gpu_pixel_batch_size` fixed; check XLA recompilation logs |
| `pmap` raises on the last batch | Batch size not divisible by device count and padding path skipped | `PmapSigmaAdamKernel.__call__` pads internally; verify the pad/unpad slice `[:n]` |
| Loss is `NaN` | $\sigma \to 0$ without the `max(sigma, 1e-6)` floor, or amplitudes carry `Inf` | Print `warmup_loss`; confirm `safe_s2` floor in `SigmaScan.per_pixel_loss` |
| Adam diverges (sigmas oscillate) | `adam_lr` too large for the normalised-profile scale | Sweep `adam_lr`; the loss curve in the figure should be smooth and convex near the optimum |

**Passing to Stage 3:** `sigma_kernel` — a compiled, callable JAX optimiser plus the bounds `sigma_lower_jax`, `sigma_upper_jax` and height axis it will be invoked with for each candidate $K$.

---
## Stage 3: Multi-$K$ Sigma Fitting

**Callable:** the kernel from Stage 2, invoked once per $K$ exactly as in `SigmaFittingExtractor._fit_all_K`.
**Input:** for each $K \in \{1,\dots,k_{\max}\}$, the first $K$ seed slots `(amps_norm, mus, sigmas_init)` from Stage 1 (amplitudes divided by each profile's peak so they live on the normalised scale), plus the normalised profiles.
**Output:** a dictionary `{K: (amps_norm, mus, sigmas_fit)}` where `sigmas_fit` has shape $(N, K)$ — the Adam-optimised spreads for a $K$-component model.

For each $K$ the kernel runs `adam_steps` of the sigma-only Adam scan, returning $\sigma^{\star}_{1:K}$. Amplitudes and centroids are passed through unchanged; only spreads move. Larger $K$ adds degrees of freedom, so its training MSE can only decrease or stay equal — the model-selection penalty in Stage 4 is what stops $K$ from always winning at $k_{\max}$.

> **What you should see:** a dict with exactly `k_max` entries keyed `1..k_max`. Entry $K$ holds three arrays each with second dimension $K$. All fitted sigmas finite and within `[sigma_lower, sigma_upper]`. The per-profile training MSE is **monotone non-increasing in $K$** (within Adam tolerance): `mse[K+1] <= mse[K] + tol` for every profile.

In [ ]:
profile_peaks       = np.maximum(synthetic_profiles.max(axis=1, keepdims=True), 1e-7).astype(np.float32)
profiles_normalised = (synthetic_profiles / profile_peaks).astype(np.float32)
profile_scales      = profile_peaks[:, 0].astype(np.float32)

adam_steps_inspect  = 400

multi_k_results = {}
for candidate_K in range(1, k_max + 1):
    amplitudes_norm_K = (initial_amplitudes[:, :candidate_K] / profile_peaks).astype(np.float32)
    centroids_K       = initial_centroids[:, :candidate_K].astype(np.float32)
    sigmas_init_K     = initial_sigmas[:, :candidate_K].astype(np.float32)

    sigmas_fit_K = np.asarray(sigma_kernel(
        jnp.asarray(sigmas_init_K),
        height_axis_jax,
        jnp.asarray(profiles_normalised),
        jnp.asarray(amplitudes_norm_K),
        jnp.asarray(centroids_K),
        sigma_lower_jax, sigma_upper_jax,
        adam_steps_inspect, float(extraction_configuration.adam_lr), float(extraction_configuration.adam_b1), float(extraction_configuration.adam_b2),
    ), dtype=np.float32)

    multi_k_results[candidate_K] = (amplitudes_norm_K, centroids_K, sigmas_fit_K)

In [ ]:
train_mse_per_k = np.zeros((synthetic_profiles.shape[0], k_max), dtype=np.float64)
for candidate_K in range(1, k_max + 1):
    amplitudes_norm_K, centroids_K, sigmas_fit_K = multi_k_results[candidate_K]
    reconstruction = GaussianMixture.evaluate_batch(synthetic_height_axis, amplitudes_norm_K, centroids_K, sigmas_fit_K)
    train_mse_per_k[:, candidate_K - 1] = ((reconstruction - profiles_normalised) ** 2).mean(axis=1)

print("multi_k_results keys     :", sorted(multi_k_results.keys()))
for candidate_K in range(1, k_max + 1):
    amplitudes_norm_K, centroids_K, sigmas_fit_K = multi_k_results[candidate_K]
    print(f"  K={candidate_K}  sigmas_fit shape={sigmas_fit_K.shape}  sigma[min/mean/max]=[{sigmas_fit_K.min():.2f}/{sigmas_fit_K.mean():.2f}/{sigmas_fit_K.max():.2f}] m  mean_mse={train_mse_per_k[:, candidate_K - 1].mean():.5f}")
print()
print("MSE monotone non-increasing per profile (tol 1e-4):")
mse_differences = np.diff(train_mse_per_k, axis=1)
print("  max upward MSE step across all profiles/K :", float(mse_differences.max()))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 5.0))

axis_left = axes[0]
for profile_index in range(synthetic_profiles.shape[0]):
    axis_left.plot(range(1, k_max + 1), train_mse_per_k[profile_index], color="0.7", lw=0.8, marker="o", ms=3, zorder=2)
axis_left.plot(range(1, k_max + 1), train_mse_per_k.mean(axis=0), color="#d62728", lw=2.0, marker="s", ms=6, label="mean over profiles", zorder=5)
axis_left.set_xlabel(r"number of Gaussians $K$")
axis_left.set_ylabel(r"normalised training MSE")
axis_left.set_xticks(range(1, k_max + 1))
axis_left.set_title(r"Stage 3 — Multi-$K$ Fitting: training MSE decreases with $K$")
axis_left.legend(fontsize=9)
axis_left.grid(True, lw=0.3, alpha=0.4)

axis_right = axes[1]
profile_to_show = 6
axis_right.plot(synthetic_height_axis, synthetic_profiles[profile_to_show], color=component_palette["data"], lw=1.6, label="data", zorder=6)
for candidate_K in (1, 3, k_max):
    amplitudes_norm_K, centroids_K, sigmas_fit_K = multi_k_results[candidate_K]
    reconstruction = GaussianMixture.evaluate_batch(synthetic_height_axis, amplitudes_norm_K[profile_to_show:profile_to_show + 1], centroids_K[profile_to_show:profile_to_show + 1], sigmas_fit_K[profile_to_show:profile_to_show + 1])[0]
    reconstruction = reconstruction * profile_scales[profile_to_show]
    axis_right.plot(synthetic_height_axis, reconstruction, lw=1.3, ls="--", label=f"$K={candidate_K}$ fit")
axis_right.set_xlabel(r"height $h$ [m]")
axis_right.set_ylabel(r"backscatter intensity")
axis_right.set_title(f"Stage 3 — Multi-$K$ Fitting: reconstructions (profile {profile_to_show}, true $K$={scatterer_count_plan[profile_to_show]})")
axis_right.legend(fontsize=9)
axis_right.grid(True, lw=0.3, alpha=0.4)

figure.tight_layout()
figure.savefig(figure_directory / "stage_3_fit_all_k.pdf")
plt.show()

In [ ]:
sigma_lower_value = float(sigma_lower_jax)
sigma_upper_value = float(sigma_upper_jax)
all_sigmas_finite = all(np.isfinite(multi_k_results[candidate_K][2]).all() for candidate_K in range(1, k_max + 1))
all_sigmas_bounded = all(
    (multi_k_results[candidate_K][2].min() >= sigma_lower_value - 1e-3) and (multi_k_results[candidate_K][2].max() <= sigma_upper_value + 1e-3)
    for candidate_K in range(1, k_max + 1)
)

stage_3_checks = [
    ("dict has k_max entries",              len(multi_k_results) == k_max),
    ("keys are 1..k_max",                   sorted(multi_k_results.keys()) == list(range(1, k_max + 1))),
    ("entry K has second-dim K",            all(multi_k_results[candidate_K][2].shape[1] == candidate_K for candidate_K in range(1, k_max + 1))),
    ("all fitted sigmas finite",            all_sigmas_finite),
    ("all fitted sigmas within bounds",     all_sigmas_bounded),
    ("training MSE monotone in K (tol)",    float(mse_differences.max()) <= 1e-3),
    ("MSE non-negative",                    float(train_mse_per_k.min()) >= 0.0),
]
for description, condition in stage_3_checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 3: Multi-$K$ Sigma Fitting

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Training MSE *rises* with $K$ | Adam not converged at low `adam_steps`, or seeds for the extra components landed in bad basins | Increase `adam_steps`; plot the MSE-vs-$K$ curve — it must be non-increasing |
| $K=k_{\max}$ MSE barely below $K=1$ | Profile is genuinely single-scatterer; extra Gaussians get near-zero amplitude | Inspect `amps_norm` for the surplus slots; this is expected, Stage 4 will pick low $K$ |
| Amplitudes change between $K$ values | Amplitudes accidentally made trainable (method is sigma-only) | Confirm only `sigmas_fit_K` differs across `K`; `amps_norm`, `mus` are passed through |
| All sigmas identical across profiles | Profiles normalised to a constant, or the kernel received the init guess as output (zero steps) | Check `adam_steps > 0` and that `profiles_normalised` varies row to row |
| Reconstruction overshoots the data peak | Amplitudes on the wrong scale (raw vs normalised) | Verify `amps_norm = amps / profile_peak` and that the figure rescales by `profile_scales` |
| MSE identical for all $K$ | The same seed slots fed to every $K$ without truncating to `:K` | Confirm the `[:, :K]` slicing of amplitudes/centroids/sigmas |

**Passing to Stage 4:** `multi_k_results` — `{K: (amps_norm, mus, sigmas_fit)}` for every candidate $K$, on the normalised intensity scale, ready to be scored against the normalised profiles.

---
## Stage 4: Best-$K$ Selection

**Callable:** `BestKSelector.select(gpu_results, prof_norm_all, scale_all, height_axis, n_params_out)`, followed conceptually by `ParameterExtractor._sort_gaussians`.
**Input:** the `{K: (amps_norm, mus, sigmas_fit)}` dict from Stage 3, the normalised profiles, the per-profile scales, the height axis, and `n_params_out = 3*k_max`.
**Output:** `best_params` of shape $(N, 3\,k_{\max})$. For each profile the winning $K^\star$ is chosen and its parameters written into the first $3K^\star$ slots (amplitudes **rescaled back** to data units); remaining slots stay zero.

The selection score per profile and per $K$ is the reconstruction MSE plus a complexity penalty proportional to $K$ and the mean amplitude:

$$\text{score}(K) \;=\; \underbrace{\frac{1}{H}\sum_n \big(\hat{y}_K(h_n) - y(h_n)\big)^2}_{\text{MSE}} \;+\; \underbrace{\lambda_K \, K \, \overline{A}}_{\text{complexity penalty}}, \qquad K^\star = \arg\min_K \text{score}(K).$$

Because the MSE is non-increasing in $K$ (Stage 3) while the penalty grows linearly, the minimum sits at the $K$ where adding a scatterer no longer reduces MSE by more than $\lambda_K \overline{A}$. The sort orders the chosen Gaussians by ascending centroid height $\mu$.

> **What you should see:** `best_params` has shape `(N, 3*k_max)`. Each profile's selected $K^\star \in \{1,\dots,k_{\max}\}$. Slots beyond $3K^\star$ are exactly zero. Amplitudes in occupied slots are back on the **data scale** (comparable to the raw profile peak, not $\le 1$). After sorting, the centroids of active components are non-decreasing within each profile. The recovered $K^\star$ should correlate with the true scatterer count used to synthesise each profile.

In [ ]:
best_k_selector = BestKSelector(k_max=k_max, lambda_k=lambda_k, logger=inspection_logger)

best_parameters = best_k_selector.select(
    gpu_results   = multi_k_results,
    prof_norm_all = profiles_normalised,
    scale_all     = profile_scales,
    height_axis   = synthetic_height_axis,
    n_params_out  = parameters_per_pixel,
)
best_parameters = best_parameters.astype(np.float32)

In [ ]:
penalised_scores = np.empty((synthetic_profiles.shape[0], k_max), dtype=np.float64)
for candidate_K in range(1, k_max + 1):
    amplitudes_norm_K, centroids_K, sigmas_fit_K = multi_k_results[candidate_K]
    reconstruction = GaussianMixture.evaluate_batch(synthetic_height_axis, amplitudes_norm_K, centroids_K, sigmas_fit_K)
    mse_K          = ((reconstruction - profiles_normalised) ** 2).mean(axis=1)
    penalty_K      = lambda_k * candidate_K * amplitudes_norm_K.mean(axis=1)
    penalised_scores[:, candidate_K - 1] = mse_K + penalty_K

selected_k_per_profile = penalised_scores.argmin(axis=1) + 1
active_component_count  = np.array([int((best_parameters[n, 0::3] >= 1e-3).sum()) for n in range(best_parameters.shape[0])])

print("best_parameters shape    :", best_parameters.shape, best_parameters.dtype)
print("n_params_out (3*k_max)   :", parameters_per_pixel)
print()
print("selected K per profile   :", selected_k_per_profile.tolist())
print("true K per profile       :", scatterer_count_plan)
print("active comp count (amp>=1e-3):", active_component_count.tolist())
print()
distribution_of_k = {int(K): int((selected_k_per_profile == K).sum()) for K in range(1, k_max + 1)}
print("best-K distribution      :", distribution_of_k)
print("amplitude slot range     :", float(best_parameters[:, 0::3].min()), "to", float(best_parameters[:, 0::3].max()), "(data scale)")

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 5.0))

axis_left = axes[0]
profile_for_curve = 7
score_curve = penalised_scores[profile_for_curve]
mse_curve   = np.array([((GaussianMixture.evaluate_batch(synthetic_height_axis, multi_k_results[K][0][profile_for_curve:profile_for_curve + 1], multi_k_results[K][1][profile_for_curve:profile_for_curve + 1], multi_k_results[K][2][profile_for_curve:profile_for_curve + 1])[0] - profiles_normalised[profile_for_curve]) ** 2).mean() for K in range(1, k_max + 1)])
penalty_curve = score_curve - mse_curve
chosen_K = int(score_curve.argmin()) + 1

axis_left.plot(range(1, k_max + 1), mse_curve, color="#1f77b4", lw=1.5, marker="o", ms=5, label="MSE")
axis_left.plot(range(1, k_max + 1), penalty_curve, color="#ff7f0e", lw=1.5, marker="^", ms=5, label=r"penalty $\lambda_K K \overline{A}$")
axis_left.plot(range(1, k_max + 1), score_curve, color="#d62728", lw=2.0, marker="s", ms=6, label="penalised score")
axis_left.scatter([chosen_K], [score_curve[chosen_K - 1]], s=160, facecolors="none", edgecolors="black", linewidths=1.6, zorder=6, label=f"$K^\\star={chosen_K}$")
axis_left.set_xlabel(r"number of Gaussians $K$")
axis_left.set_ylabel("score (normalised units)")
axis_left.set_xticks(range(1, k_max + 1))
axis_left.set_title(f"Stage 4 — Best-$K$: penalised score (profile {profile_for_curve}, true $K$={scatterer_count_plan[profile_for_curve]})")
axis_left.legend(fontsize=9)
axis_left.grid(True, lw=0.3, alpha=0.4)

axis_right = axes[1]
k_values   = list(range(1, k_max + 1))
selected_counts = [distribution_of_k[K] for K in k_values]
true_counts     = [int(np.sum(np.array(scatterer_count_plan) == K)) for K in k_values]
bar_width = 0.4
axis_right.bar([k - bar_width / 2 for k in k_values], true_counts, width=bar_width, color="#2ca02c", alpha=0.8, edgecolor="white", label="true $K$")
axis_right.bar([k + bar_width / 2 for k in k_values], selected_counts, width=bar_width, color="#d62728", alpha=0.8, edgecolor="white", label="selected $K^\\star$")
axis_right.set_xlabel(r"number of scatterers $K$")
axis_right.set_ylabel("number of profiles")
axis_right.set_xticks(k_values)
axis_right.set_title("Stage 4 — Best-$K$: selected vs true scatterer counts")
axis_right.legend(fontsize=9)
axis_right.grid(True, axis="y", lw=0.3, alpha=0.4)

figure.tight_layout()
figure.savefig(figure_directory / "stage_4_best_k.pdf")
plt.show()

In [ ]:
centroids_sorted_ok = True
for profile_index in range(best_parameters.shape[0]):
    active_mus = [best_parameters[profile_index, 3 * slot + 1] for slot in range(k_max) if best_parameters[profile_index, 3 * slot] >= 1e-3]
    if any(active_mus[i] > active_mus[i + 1] + 1e-3 for i in range(len(active_mus) - 1)):
        centroids_sorted_ok = False
        break

zero_slots_beyond_kstar = True
for profile_index in range(best_parameters.shape[0]):
    selected_K = int(selected_k_per_profile[profile_index])
    if not np.allclose(best_parameters[profile_index, 3 * selected_K:], 0.0):
        active_extra = int((best_parameters[profile_index, 3 * selected_K::3] >= 1e-3).sum())
        if active_extra > 0:
            zero_slots_beyond_kstar = False
            break

stage_4_checks = [
    ("best_params shape (N, 3*k_max)",      best_parameters.shape == (synthetic_profiles.shape[0], parameters_per_pixel)),
    ("dtype float32",                       best_parameters.dtype == np.float32),
    ("no NaN or Inf",                       bool(np.isfinite(best_parameters).all())),
    ("selected K within 1..k_max",          int(selected_k_per_profile.min()) >= 1 and int(selected_k_per_profile.max()) <= k_max),
    ("amplitudes back on data scale",       float(best_parameters[:, 0::3].max()) > 1.0 + 1e-3 or float(synthetic_profiles.max()) <= 1.0),
    ("inactive slots beyond K* are zero",   zero_slots_beyond_kstar),
    ("active centroids non-decreasing",     centroids_sorted_ok),
    ("sigmas non-negative",                 float(best_parameters[:, 2::3].min()) >= 0.0),
]
for description, condition in stage_4_checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 4: Best-$K$ Selection

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Every profile selects $K^\star = k_{\max}$ | `lambda_k` too small — penalty never overtakes the MSE gain | Plot the penalty vs MSE curves; raise `lambda_k` until the score has an interior minimum |
| Every profile selects $K^\star = 1$ | `lambda_k` too large — penalty dwarfs MSE at $K>1$ | Same plot; lower `lambda_k` so genuine multi-scatterer profiles win higher $K$ |
| Amplitudes $\le 1$ in `best_params` | Forgot to rescale `amps_norm * scale_all` back to data units | Compare `best_params[:, 0::3].max()` against the raw profile peak |
| Centroids not sorted within a pixel | `_sort_gaussians` not applied, or sort key ignored inactive (amp $\le 10^{-3}$) slots | Check active `mu` ordering per profile; inactive slots are pushed to $+\infty$ by design |
| Non-zero junk in slots beyond $3K^\star$ | Wrong `np.ix_` index ranges when writing the winning block | Confirm slots `3*K*: ` are all zero for the selected $K$ |
| Selected $K^\star$ uncorrelated with true count | Peak init seeded poorly (Stage 1), so high-$K$ models never fit better | Cross-check the selected-vs-true bar chart; if flat, revisit Stage 1 seeds |

**Passing to Stage 5:** `best_parameters` — the per-pixel parameter vector `(N, 3*k_max)` on the data scale, height-sorted; reshaped to `(3*k_max, Az, R)` it is the parameter stack the metrics calculator consumes.

---
## Stage 5: Fitting Metrics

**Callable:** `FittingMetricsCalculator.run(parameters_array, metadata, tomogram_path)`.
**Input:** the parameter stack of shape $(3\,k_{\max}, A_z, R)$, a metadata dict carrying `height_range`, and the path to the tomogram cube.
**Output:** a metrics dict with `r2_map` $(A_z, R)$, `activity_map` $(A_z, R)$, `height_axis` $(H,)$, a `global_summary` dict, and per-Gaussian `amp_k` / `mu_k` / `sigma_k` maps plus adjacent `mu_sep_k` maps.

The per-pixel coefficient of determination compares the reconstructed profile against the tomogram profile across all height bins:

$$R^2 \;=\; 1 - \frac{\sum_j \big(\hat{y}(h_j) - y(h_j)\big)^2}{\sum_j \big(y(h_j) - \bar{y}\big)^2 + \varepsilon},$$

and the activity map counts active scatterers per pixel, $K_{\text{active}} = \sum_{k} \mathbb{1}[A_k \ge 10^{-3}]$. Here we drive the calculator on a small synthetic cube assembled from the Stage 4 parameters so the call is fully self-contained.

> **What you should see:** `r2_map` and `activity_map` both shape `(Az, R)`. $R^2 \le 1$ everywhere; for well-fit synthetic pixels the median $R^2$ should sit close to 1. `activity_map` values in `{0,...,k_max}` and match each pixel's selected $K^\star$. `global_summary` carries `r2_mean`, `r2_median`, percentile fields, `r2_neg_frac`, and `frac_K_active` fractions that sum to 1 across $K$.

In [ ]:
synthetic_azimuth = 3
synthetic_range   = 4
assert synthetic_azimuth * synthetic_range == synthetic_profiles.shape[0]

parameter_stack = best_parameters.reshape(synthetic_azimuth, synthetic_range, parameters_per_pixel).transpose(2, 0, 1).astype(np.float32)

synthetic_cube = np.zeros((synthetic_height_bins, synthetic_azimuth, synthetic_range), dtype=np.float32)
for height_index in range(synthetic_height_bins):
    synthetic_cube[height_index] = GaussianMixture.evaluate_slice(parameter_stack, float(synthetic_height_axis[height_index]), k_max)

synthetic_cube_path = figure_directory / "synthetic_tomogram.npy"
np.save(str(synthetic_cube_path), synthetic_cube, allow_pickle=False)

metrics_calculator = FittingMetricsCalculator(n_gaussians=k_max, logger=inspection_logger)
metrics_metadata   = {"height_range": [synthetic_height_low, synthetic_height_high]}

metrics_result = metrics_calculator.run(parameter_stack, metrics_metadata, synthetic_cube_path)

In [ ]:
r2_map        = metrics_result["r2_map"]
activity_map  = metrics_result["activity_map"]
global_summary = metrics_result["global_summary"]

print("metrics_result keys      :", sorted(metrics_result.keys()))
print()
print("r2_map shape             :", r2_map.shape, r2_map.dtype)
print("activity_map shape       :", activity_map.shape, activity_map.dtype)
print("r2 min / median / max    :", float(np.nanmin(r2_map)), float(np.nanmedian(r2_map)), float(np.nanmax(r2_map)))
print("activity unique values   :", np.unique(activity_map))
print()
print("global_summary:")
for summary_key in ["n_pixels", "n_gaussians", "r2_mean", "r2_median", "r2_std", "r2_p25", "r2_p50", "r2_p75", "r2_neg_frac"]:
    print(f"  {summary_key:<14}: {global_summary.get(summary_key)}")
active_fraction_sum = sum(global_summary.get(f"frac_{K}_active", 0.0) for K in range(k_max + 1))
print("  frac_*_active sum :", active_fraction_sum)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

axis_left = axes[0]
image_r2  = axis_left.imshow(r2_map, cmap="RdYlGn", vmin=max(-1.0, float(np.nanpercentile(r2_map, 1.0))), vmax=1.0, aspect="auto", interpolation="nearest")
axis_left.set_xlabel("range [px]")
axis_left.set_ylabel("azimuth [px]")
axis_left.set_title(r"Stage 5 — Metrics: per-pixel $R^2$ map")
colorbar_r2 = figure.colorbar(image_r2, ax=axis_left, fraction=0.046, pad=0.04)
colorbar_r2.set_label(r"$R^2$")

axis_right = axes[1]
activity_levels = list(range(k_max + 2))
image_activity  = axis_right.imshow(activity_map, cmap="viridis", vmin=0, vmax=k_max, aspect="auto", interpolation="nearest")
axis_right.set_xlabel("range [px]")
axis_right.set_ylabel("azimuth [px]")
axis_right.set_title(r"Stage 5 — Metrics: active-Gaussian count $K_{\mathrm{active}}$")
colorbar_activity = figure.colorbar(image_activity, ax=axis_right, fraction=0.046, pad=0.04, ticks=list(range(k_max + 1)))
colorbar_activity.set_label(r"active Gaussians $K$")

figure.tight_layout()
figure.savefig(figure_directory / "stage_5_metrics.pdf")
plt.show()

In [ ]:
active_fraction_keys   = [f"frac_{K}_active" for K in range(k_max + 1)]
active_fraction_total  = sum(global_summary.get(key, 0.0) for key in active_fraction_keys)

stage_5_checks = [
    ("r2_map shape (Az, R)",                r2_map.shape == (synthetic_azimuth, synthetic_range)),
    ("activity_map shape (Az, R)",          activity_map.shape == (synthetic_azimuth, synthetic_range)),
    ("R2 <= 1 everywhere",                  float(np.nanmax(r2_map)) <= 1.0 + 1e-5),
    ("activity within 0..k_max",            int(activity_map.min()) >= 0 and int(activity_map.max()) <= k_max),
    ("per-Gaussian maps present",           all(f"amp_{k}" in metrics_result and f"mu_{k}" in metrics_result and f"sigma_{k}" in metrics_result for k in range(k_max))),
    ("summary n_gaussians == k_max",        int(global_summary["n_gaussians"]) == k_max),
    ("summary n_pixels == Az*R",            int(global_summary["n_pixels"]) == synthetic_azimuth * synthetic_range),
    ("active fractions sum to 1",           abs(active_fraction_total - 1.0) < 1e-5),
    ("R2 median finite",                    np.isfinite(global_summary["r2_median"])),
]
for description, condition in stage_5_checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 5: Fitting Metrics

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| $R^2$ uniformly near 1 even for bad fits | Metrics computed against the reconstruction itself, not the real tomogram | Confirm `tomogram_path` points at measured data, not the synthesised cube, in production |
| Large negative $R^2$ fraction | Centroids/spreads in wrong units (height axis mismatch between fitting and metrics) | Compare `metadata["height_range"]` here with the range used in Stage 1 |
| `activity_map` disagrees with selected $K^\star$ | `amp_threshold` differs from the $10^{-3}$ used in Stage 4 selection | Print `metrics_calculator.amp_threshold`; keep it consistent across stages |
| Per-Gaussian maps all `NaN` | Every amplitude below threshold — degenerate (empty) fit | Inspect `amp_k` maps; all-NaN means no active scatterers anywhere |
| `frac_K_active` fractions do not sum to 1 | Activity counted a value outside `0..k_max` | Verify `activity_map` range; the count loop must cover `k_max + 1` bins |
| $R^2$ map shape transposed | Parameter stack reshaped as `(R, Az)` instead of `(Az, R)` | Check `parameter_stack.shape[1:]` equals `(Az, R)` |

**Passing to Stage 6:** `parameter_stack` — the `(3*k_max, Az, R)` array, plus `metrics_result`; the stack is what gets serialised, and the metrics dict feeds the (separately tested) plotting component.

---
## Stage 6: Artifact I/O — Parameter Stack and Run Metadata

**Callable:** `ParameterIO.save_params(parameters_array, npy_path)` / `ParameterIO.load_params(npy_path)` / `ParameterIO.load_metadata(meta_path)`, and `ExtractionMetadataManager.save_run_metadata(npy_path, tomogram_path, height_range)`.
**Input:** the parameter stack `(3*k_max, Az, R)` and the config-derived output paths.
**Output:** a `.npy` file holding the contiguous `float32` parameter stack, and a `param_extraction_meta.json` file with the run provenance (timestamp, source tomogram, height range, `number_of_gaussians`, output paths).

This stage is a faithful round-trip: there is no mathematical transformation, only serialisation. The contract is that what is loaded back equals bit-for-bit what was saved, and that the metadata records exactly the configuration used so a downstream metrics/plot run can reconstruct the context.

> **What you should see:** the saved `.npy` round-trips to an array identical to `parameter_stack` with the same shape `(3*k_max, Az, R)` and `float32` dtype. The metadata JSON parses to a dict whose `number_of_gaussians == k_max`, whose `height_range` equals the configured range, and whose `parameters_npy` matches the saved file name.

In [ ]:
parameter_io = ParameterIO(logger=inspection_logger)

artifact_directory = figure_directory / "artifacts"
artifact_directory.mkdir(parents=True, exist_ok=True)
parameters_npy_path = artifact_directory / "parameters_inspection.npy"

saved_npy_path = parameter_io.save_params(parameter_stack, parameters_npy_path)
loaded_parameter_stack = parameter_io.load_params(saved_npy_path)

metadata_payload = {
    "timestamp"           : "inspection",
    "source_tomogram"     : str(synthetic_cube_path),
    "height_range"        : [synthetic_height_low, synthetic_height_high],
    "output_directory"    : str(artifact_directory),
    "parameters_npy"      : saved_npy_path.name,
    "number_of_gaussians" : k_max,
}
metadata_json_path = artifact_directory / "param_extraction_meta.json"
with open(metadata_json_path, "w", encoding="utf-8") as metadata_file:
    json.dump(metadata_payload, metadata_file, indent=4)

loaded_metadata = parameter_io.load_metadata(metadata_json_path)

In [ ]:
print("saved npy path           :", saved_npy_path)
print("file exists               :", saved_npy_path.is_file())
print("file size (bytes)         :", saved_npy_path.stat().st_size if saved_npy_path.is_file() else None)
print()
print("loaded stack shape        :", loaded_parameter_stack.shape, loaded_parameter_stack.dtype)
print("round-trip exact match    :", bool(np.array_equal(loaded_parameter_stack, parameter_stack)))
print()
print("metadata json path        :", metadata_json_path)
print("loaded metadata keys      :", sorted(loaded_metadata.keys()))
for metadata_key in ["height_range", "number_of_gaussians", "parameters_npy", "source_tomogram"]:
    print(f"  {metadata_key:<20}: {loaded_metadata.get(metadata_key)}")

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

axis_left = axes[0]
roundtrip_difference = np.abs(loaded_parameter_stack - parameter_stack).reshape(parameters_per_pixel, -1).max(axis=1)
axis_left.bar(range(parameters_per_pixel), roundtrip_difference, color="#1f77b4", edgecolor="white", linewidth=0.4)
axis_left.set_xlabel("parameter channel index")
axis_left.set_ylabel(r"max $|$loaded $-$ saved$|$")
axis_left.set_title("Stage 6 — Artifact I/O: round-trip channel-wise error")
axis_left.grid(True, axis="y", lw=0.3, alpha=0.4)

axis_right = axes[1]
channel_groups = ["amplitude $A_k$", "centroid $\\mu_k$", "spread $\\sigma_k$"]
group_means    = [float(np.nanmean(np.where(loaded_parameter_stack[offset::3] >= 1e-3, loaded_parameter_stack[offset::3], np.nan))) if offset == 0 else float(np.nanmean(np.where(loaded_parameter_stack[0::3] >= 1e-3, loaded_parameter_stack[offset::3], np.nan))) for offset in range(3)]
group_colors   = ["#1f77b4", "#2ca02c", "#9467bd"]
axis_right.bar(channel_groups, group_means, color=group_colors, edgecolor="white", linewidth=0.5)
axis_right.set_ylabel("mean over active components")
axis_right.set_title("Stage 6 — Artifact I/O: persisted parameter group means")
axis_right.grid(True, axis="y", lw=0.3, alpha=0.4)

figure.tight_layout()
figure.savefig(figure_directory / "stage_6_artifact_io.pdf")
plt.show()

In [ ]:
stage_6_checks = [
    ("npy file written to disk",            saved_npy_path.is_file()),
    ("loaded shape matches saved",          loaded_parameter_stack.shape == parameter_stack.shape),
    ("loaded dtype float32",                loaded_parameter_stack.dtype == np.float32),
    ("round-trip is bit-exact",             bool(np.array_equal(loaded_parameter_stack, parameter_stack))),
    ("metadata json parses to dict",        isinstance(loaded_metadata, dict)),
    ("metadata number_of_gaussians == k_max", int(loaded_metadata["number_of_gaussians"]) == k_max),
    ("metadata height_range matches",       list(loaded_metadata["height_range"]) == [synthetic_height_low, synthetic_height_high]),
    ("metadata parameters_npy matches file", loaded_metadata["parameters_npy"] == saved_npy_path.name),
]
for description, condition in stage_6_checks:
    print(f"[{'PASS' if condition else 'FAIL'}] {description}")

### Common Mistakes — Stage 6: Artifact I/O

| Symptom | Likely Cause | How to Diagnose |
|---------|-------------|-----------------|
| Round-trip not bit-exact | Stack not made contiguous, or saved with `allow_pickle=True` and re-cast | `save_params` calls `np.ascontiguousarray`; confirm dtype stays `float32` |
| Loaded array shape transposed | Saved before the `(3*k_max, Az, R)` transpose was applied | Print `loaded_parameter_stack.shape`; first axis must be `3*k_max` |
| Metadata `height_range` `null` | `discover_height_range` found no config state and no explicit range | Set `ExtractionConfig.height_range` explicitly, or verify the meta directory |
| `number_of_gaussians` mismatched with stack | Metadata written with a different `k_max` than the fit used | Cross-check `loaded_metadata["number_of_gaussians"]` against `parameter_stack.shape[0] // 3` |
| Downstream load fails | `.npy` path differs from `parameters_npy_path` in config | Confirm `saved_npy_path == config.parameters_npy_path` in a real run |
| Metadata path collides between runs | Two runs share an `output_directory` (same `output_suffix_value`) | Vary `output_suffix` or `k_max` so `output_subdir_name` differs |

**End of pipeline:** `saved_npy_path` and `metadata_json_path` are the two artifacts the parameter-extraction pipeline produces; the metrics and plotting components (Stage 5 and the separately-tested plotter) read them back to render the publication figures.

---
## End-to-End Summary

The cells below run the full `ParamExtractionPipeline.run()` orchestrator when a real tomogram and processed-data layout are present on disk, then assert that the artifacts it produces are consistent with the stage-by-stage results assembled above. When no real data is available the orchestration block is skipped (it requires the discovered tomogram and GPU), but the stage-by-stage assertions already cover every component contract.

A structured summary table of every intermediate output's shape, type, and key statistic is printed first so the whole information flow is visible at a glance.

In [ ]:
summary_rows = [
    ("Stage 1  peak init",        "(amps, mus, sigmas)", f"{initial_amplitudes.shape} x3", f"mu range [{initial_centroids.min():.1f}, {initial_centroids.max():.1f}] m"),
    ("Stage 2  sigma kernel",     type(sigma_kernel).__name__, f"warmup {warmup_sigmas_fit.shape}", f"loss {warmup_loss:.4f}"),
    ("Stage 3  multi-K fitting",  "dict[K -> arrays]", f"{len(multi_k_results)} entries", f"mean MSE@k_max {train_mse_per_k[:, -1].mean():.4f}"),
    ("Stage 4  best-K select",    "best_params", f"{best_parameters.shape}", f"K* dist {distribution_of_k}"),
    ("Stage 5  metrics",          "metrics dict", f"r2 {r2_map.shape}", f"median R2 {global_summary['r2_median']:.3f}"),
    ("Stage 6  artifact io",      "npy + json", f"{loaded_parameter_stack.shape}", f"bit-exact {bool(np.array_equal(loaded_parameter_stack, parameter_stack))}"),
]

column_headers = ("stage", "output type", "shape / size", "key statistic")
print(f"{column_headers[0]:<24}{column_headers[1]:<22}{column_headers[2]:<22}{column_headers[3]}")
print("-" * 100)
for stage_label, output_type, shape_text, statistic_text in summary_rows:
    print(f"{stage_label:<24}{output_type:<22}{shape_text:<22}{statistic_text}")

In [ ]:
run_real_pipeline = extraction_configuration.discover_tomogram_path() is not None

if run_real_pipeline:
    param_extraction_pipeline = ParamExtractionPipeline(extraction_configuration, logger=inspection_logger)
    pipeline_artifacts        = param_extraction_pipeline.run()

    full_parameter_stack = np.load(str(pipeline_artifacts["parameters_npy"])).astype(np.float32)

    end_to_end_checks = [
        ("parameters_npy artifact exists",      pipeline_artifacts["parameters_npy"].is_file()),
        ("metadata artifact exists",            pipeline_artifacts["metadata"].is_file()),
        ("stack first axis is 3*k_max",         full_parameter_stack.shape[0] == parameters_per_pixel),
        ("stack dtype float32",                 full_parameter_stack.dtype == np.float32),
        ("all amplitudes non-negative",         float(full_parameter_stack[0::3].min()) >= 0.0),
        ("plot artifacts returned",             isinstance(pipeline_artifacts["plots"], dict) and len(pipeline_artifacts["plots"]) > 0),
    ]
    for description, condition in end_to_end_checks:
        print(f"[{'PASS' if condition else 'FAIL'}] {description}")
else:
    print("No tomogram discovered under", extraction_configuration.data_directory)
    print("Skipping full-orchestrator run; per-stage assertions above cover every component contract.")
    print("To run end to end, place a tomogram cube in the processed-data 'data' directory and re-execute this cell.")

### Verification Outcome

Every stage above is bracketed by structural → statistical → semantic assertions:

- **Structural:** correct shapes (`(N, k_max)` seeds, `(N, 3*k_max)` best params, `(Az, R)` maps), correct dtypes (`float32`), correct dict keys.
- **Statistical:** all values finite; sigmas within `[dh, span/2]`; $R^2 \le 1$; non-negative amplitudes and spreads.
- **Semantic:** training MSE monotone in $K$; selected $K^\star$ tracks the true scatterer count; centroids height-sorted; active-Gaussian fractions sum to 1; artifact round-trip bit-exact.

Run this notebook before and after any change to a `param_pipeline` component: if the assertions still pass, the change preserved the contract; if one fails, it names the exact stage and invariant that broke.